# The InMAP ISRM on EarthSciLab

Two `.esm` documents run through the [EarthSciLab](https://earthscilab.com) API,
each drawn as a map of the 52,411-cell InMAP receptor grid.

| section | document | emissions |
|---|---|---|
| 1 | — | setup: the API client, and signing in |
| 2 | `isrm_point.esm` | the EPA 2016fd EGU point inventory, with plume rise |
| 3 | `isrm_polygon.esm` | an example county-polygon layer, allocated by area overlap |

**This is not a local run.** `run-rs`, `run-jl` and `run-py` drive `earthsci-ast`
in-process and need the engine, EarthSciIO, a 69 MB EPA zip and a multi-GB chunk
cache to do it. Here EarthSciLab's own hardware fetches the ~14 GB of
source–receptor slabs and what arrives is 52,411 numbers per variable.

**About the maps.** Both documents declare a `scatter` plot, because the ISRM
grid is an unstructured variable-resolution polygon mesh with a single flat
index and a true choropleth is out of the `.esm` plot vocabulary's scope
(gap G5). A notebook is not bound by that: both documents also compute
`rcv_W`, `rcv_S`, `rcv_E` and `rcv_N` — every cell's own rectangle in
Lambert-conformal metres — so `draw_map` below asks for those alongside the
concentration and draws the real mesh.

Needs `matplotlib`; section 3 also needs `pyshp` to build its example layer.

```
pip install -r requirements.txt
```

---
# 1. Setup

Everything the other two sections use. Standard library plus `matplotlib`.

[`run_isrm_demo.py`](run_isrm_demo.py) beside this notebook is the same thing as
a command-line script for the point document; this is deliberately self-contained
so the notebook can travel on its own.

In [ ]:
import base64
import io
import json
import os
import ssl
import subprocess
import sys
import time
import urllib.error
import urllib.parse
import urllib.request
import webbrowser
from pathlib import Path

def _find_repo(start=None):
    """The isrm.esm checkout, found by looking for the documents themselves.

    Not derived from the working directory: VS Code starts a notebook's kernel in
    whatever `jupyter.notebookFileRoot` says (the notebook's own folder by
    default, the workspace root if configured), and Jupyter Lab starts it
    wherever it was launched. Walking up for `isrm_point.esm` is right under all
    of them, and says so plainly when it is wrong.
    """
    here = Path(start or os.getcwd()).resolve()
    for candidate in (here, *here.parents):
        if (candidate / "isrm_point.esm").is_file():
            return candidate
    raise FileNotFoundError(
        f"no isrm_point.esm at or above {here}; open this notebook from inside the "
        "isrm.esm checkout, or set REPO by hand")


REPO = _find_repo()
API = os.environ.get("EARTHSCILAB_API", "https://api.earthscilab.com")
WORKOS = "https://api.workos.com"
CREDENTIALS = Path(os.environ.get(
    "EARTHSCILAB_CREDENTIALS", os.path.expanduser("~/.earthscilab/credentials.json")))

# `GET /datasets/{id}/field` clamps to this and a query cannot raise it. The
# receptor axis is 52,411, so a whole field fits and comes back at stride 1.
MAX_VALUES = 262_144

# The cell-rectangle observeds every map needs, over and above the value drawn.
GEOMETRY = ["rcv_W", "rcv_S", "rcv_E", "rcv_N"]

print("repo:", REPO)
print("api: ", API)

## 1.1 HTTP

Two error types, because they are answered differently: an `HttpError` from
EarthSciLab is something to report, while a 400 from WorkOS carrying an OAuth
`error` code is part of the device-grant protocol — `authorization_pending` is
the normal case, not a failure.

In [ ]:
class HttpError(Exception):
    def __init__(self, status, body, url):
        super().__init__(f"HTTP {status} from {url}: {body[:600]}")
        self.status, self.body = status, body


class OAuthError(Exception):
    """A 400 from WorkOS carrying an OAuth 2.0 `error` code."""
    def __init__(self, error, description):
        super().__init__(f"{error}: {description}")
        self.error = error


def _open(req, timeout):
    try:
        return urllib.request.urlopen(req, timeout=timeout, context=ssl.create_default_context())
    except urllib.error.HTTPError as e:
        body = e.read().decode("utf-8", "replace")
        try:
            payload = json.loads(body)
        except ValueError:
            payload = {}
        if e.code == 400 and "error" in payload:
            raise OAuthError(payload["error"], payload.get("error_description", "")) from None
        raise HttpError(e.code, body, req.full_url) from None


def http_json(method, url, *, json_body=None, form=None, raw=None, headers=None, timeout=120.0):
    """One request, JSON in and JSON out (or `None` for an empty 204 body)."""
    data, hdrs = None, dict(headers or {})
    if json_body is not None:
        data = json.dumps(json_body).encode()
        hdrs["Content-Type"] = "application/json"
    elif form is not None:
        data = urllib.parse.urlencode(form).encode()
        hdrs["Content-Type"] = "application/x-www-form-urlencoded"
    elif raw is not None:
        data = raw
        hdrs["Content-Type"] = "application/octet-stream"
    req = urllib.request.Request(url, data=data, headers=hdrs, method=method)
    with _open(req, timeout) as resp:
        body = resp.read()
    return json.loads(body) if body else None

## 1.2 Signing in

**The OAuth 2.0 device authorization grant** — WorkOS's "CLI Auth", which AuthKit
serves with no extra configuration. It is the right flow here for a specific
reason rather than a stylistic one: a full-scale run takes about an hour and a
WorkOS *access* token is short-lived, so a hand-pasted token expires long before
the answer exists. The device grant hands back a **refresh** token, which this
class stores (mode 0600), rotates on every use, and spends to mint a fresh access
token before each request. What it produces is an ordinary AuthKit user JWT —
same JWKS, same `sub` — so the API needed no change to accept it and runs bill to
your own account.

In [ ]:
class Session:
    """A signed-in EarthSciLab caller, refreshed on demand.

    The reason this is a class and not a header constant: "the token" is a thing
    that has to be re-derived, not held. Every request goes through `headers()`,
    which mints a new one whenever the current one is within a minute of expiry.
    """

    DEVICE_GRANT = "urn:ietf:params:oauth:grant-type:device_code"

    def __init__(self, api=API):
        self.api = api
        self.client_id = http_json("GET", f"{api}/auth/config")["client_id"]
        self._store = self._load()
        self._access = self._store.get("access_token")

    def _load(self):
        try:
            return json.loads(CREDENTIALS.read_text()).get(self.api, {})
        except (OSError, ValueError):
            return {}

    def _save(self):
        try:
            everything = json.loads(CREDENTIALS.read_text())
        except (OSError, ValueError):
            everything = {}
        everything[self.api] = self._store
        CREDENTIALS.parent.mkdir(parents=True, exist_ok=True)
        fd = os.open(CREDENTIALS, os.O_WRONLY | os.O_CREAT | os.O_TRUNC, 0o600)
        with os.fdopen(fd, "w") as fh:
            json.dump(everything, fh, indent=1)

    @staticmethod
    def _expiry(token):
        """`exp` out of a JWT, read WITHOUT verifying it.

        Reading a claim to decide when to refresh is not the same act as trusting
        one: the API verifies this token against the JWKS, and a lie here can only
        cost an unnecessary refresh.
        """
        if not token:
            return 0.0
        try:
            payload = token.split(".")[1]
            payload += "=" * (-len(payload) % 4)
            return float(json.loads(base64.urlsafe_b64decode(payload)).get("exp", 0))
        except Exception:
            return 0.0

    def _adopt(self, response):
        self._access = response["access_token"]
        self._store["access_token"] = self._access
        # Refresh tokens ROTATE. Persisting the new one is not housekeeping —
        # keep the old one and the next session has to sign in again.
        if response.get("refresh_token"):
            self._store["refresh_token"] = response["refresh_token"]
        self._save()

    def _refresh(self):
        token = self._store.get("refresh_token")
        if not token:
            return False
        try:
            self._adopt(http_json("POST", f"{WORKOS}/user_management/authenticate", form={
                "grant_type": "refresh_token", "refresh_token": token,
                "client_id": self.client_id}))
            return True
        except OAuthError:
            self._store.pop("refresh_token", None)
            return False

    def login(self):
        start = http_json("POST", f"{WORKOS}/user_management/authorize/device",
                          form={"client_id": self.client_id})
        print(f"\n  Your code is:  {start['user_code']}")
        print(f"  Open: {start['verification_uri_complete']}\n")
        try:
            webbrowser.open(start["verification_uri_complete"])
        except Exception:
            pass
        interval = float(start.get("interval", 5))
        deadline = time.time() + float(start.get("expires_in", 300))
        while time.time() < deadline:
            time.sleep(interval)
            try:
                self._adopt(http_json("POST", f"{WORKOS}/user_management/authenticate", form={
                    "grant_type": self.DEVICE_GRANT, "device_code": start["device_code"],
                    "client_id": self.client_id}))
                return
            except OAuthError as e:
                if e.error == "authorization_pending":
                    continue
                if e.error == "slow_down":
                    interval += 1
                    continue
                raise RuntimeError(f"sign-in refused: {e}") from None
        raise RuntimeError("sign-in timed out; run this cell again")

    def headers(self):
        if time.time() > self._expiry(self._access) - 60:
            if not self._refresh():
                self.login()
        return {"Authorization": f"Bearer {self._access}"}

    def get(self, path, timeout=120.0):
        return http_json("GET", f"{self.api}{path}", headers=self.headers(), timeout=timeout)

    def post(self, path, body=None, timeout=300.0):
        return http_json("POST", f"{self.api}{path}", json_body=body,
                         headers=self.headers(), timeout=timeout)

    def put_bytes(self, path, payload, timeout=600.0):
        return http_json("PUT", f"{self.api}{path}", raw=payload,
                         headers=self.headers(), timeout=timeout)

    def stream(self, path, timeout):
        headers = dict(self.headers(), Accept="text/event-stream")
        req = urllib.request.Request(f"{self.api}{path}", headers=headers, method="GET")
        return _open(req, timeout)

In [ ]:
session = Session()
me = session.get("/me")
print("signed in as", me["email"])
print("credit:", session.get("/credits"))

## 1.3 Preparing a document

Three edits stand between a `.esm` on disk and one a dispatched run can read.
Each is a run that fails without it.

**`inline_template_library`.** Both documents reach their shared body by
`{"ref": "./isrm_base.esm"}`, and a document that arrives over the wire has no
directory to be relative to — the server anchors relative refs at *its* working
directory, so the raw document fails on the runner with `template-library file
not found: isrm_base.esm`. Merging the library's `expression_templates` into the
importing model and dropping the import key is the whole fix.

This is deliberately **not** a full esm-spec §9.7 template resolve, which would
also *close* the metaparameters — binding `N_REC` to its declared default of `0`
before the loader has discovered how many records there are. That run dies inside
the engine on a zero-length axis. `N_REC` is the documents' own G6: a
metaparameter only the loader knows the value of, so the declaration has to reach
the engine still open.

**`truncate_records`.** The scale knob is a *document* edit, not a request
parameter — a loader-level `select` range (esm-spec §8.9.2) on every source that
discovers its own extent. Because the selection follows the loader's own
`record_filter`, `extent` re-discovers the smaller `N_REC` by itself and nothing
else changes. A run's scale is a property of the request, and requests carry
documents.

**`observeds_for`.** What the run must compute. The names live in the *document* —
`metadata.x_esd.report` names the totals, and the analysis names its plot's
variables — which is what lets one function serve both geometries without a table
of either. Named observeds must be producible or the run fails **by name**; that
is the point, since the "give me everything" mode silently skips an observed it
cannot evaluate.

In [ ]:
def load_document(path):
    with open(path) as fh:
        return json.load(fh)


def inline_template_library(doc, base):
    """Merge every `expression_template_imports` target into its importer."""
    inlined = []
    for model in (doc.get("models") or {}).values():
        refs = [i["ref"] for i in model.get("expression_template_imports", []) if "ref" in i]
        if not refs:
            continue
        merged = {}
        for ref in refs:
            path = os.path.normpath(os.path.join(base, ref))
            with open(path) as fh:
                merged.update(json.load(fh).get("expression_templates") or {})
            inlined.append(os.path.basename(path))
        model.pop("expression_template_imports", None)
        merged.update(model.get("expression_templates") or {})   # the document's own win
        model["expression_templates"] = merged
    return inlined


def truncate_records(doc, n):
    """Keep only the first `n` records of every self-measuring loader."""
    touched = []
    for name, source in (doc.get("data_sources") or {}).items():
        if (source.get("extent") or {}).get("metaparameter"):
            source["select"] = {"axes": [{"range": {"start": 0, "stop": n}}]}
            touched.append(name)
    return touched


def find_analysis(doc, analysis_id):
    for model_name, model in (doc.get("models") or {}).items():
        for analysis in model.get("analyses") or []:
            if analysis.get("id") == analysis_id:
                return model_name, analysis
    have = [a.get("id") for m in (doc.get("models") or {}).values()
            for a in (m.get("analyses") or [])]
    raise KeyError(f"no analysis {analysis_id!r}; this document declares {have}")


def totals_for(doc):
    """The sums to report, named by the document rather than by this notebook.

    `metadata.x_esd.report` exists so a runner can drive either geometry without
    carrying a table of either: `total_pm25` names one observed, and `deaths`
    maps each concentration-response function to its own.
    """
    report = ((doc.get("metadata") or {}).get("x_esd") or {}).get("report") or {}
    named = {}
    if report.get("total_pm25"):
        named["total_pm25"] = report["total_pm25"]
    for function, observed in (report.get("deaths") or {}).items():
        named[f"deaths/{function}"] = observed
    return named


def observeds_for(doc, analysis, extra=GEOMETRY):
    """Totals + the analysis's own plot variables + whatever the map needs."""
    wanted = list(dict.fromkeys(totals_for(doc).values()))
    for plot in analysis.get("plots") or []:
        for axis in ("x", "y"):
            var = (plot.get(axis) or {}).get("variable")
            if var:
                wanted.append(var)
    wanted.extend(extra)
    return list(dict.fromkeys(wanted))


def prepare(path, records=None, analysis_id=None):
    """A document ready to send, plus the analysis and the observeds it needs."""
    doc = load_document(path)
    model_name, analysis = find_analysis(doc, analysis_id) if analysis_id else (
        next(iter(doc["models"])), (next(iter(doc["models"].values()))["analyses"] or [{}])[0])
    inlined = inline_template_library(doc, os.path.dirname(os.path.abspath(path)))
    truncated = truncate_records(doc, records) if records else []
    observeds = observeds_for(doc, analysis)
    print(f"{os.path.basename(path)} :: {model_name} :: {analysis.get('id')}")
    print(f"  inlined    {', '.join(inlined) or '(nothing)'}")
    print(f"  scale      {f'first {records:,} records of ' + ', '.join(truncated) if records else 'FULL'}")
    print(f"  observeds  {', '.join(observeds)}")
    return doc, analysis, observeds

## 1.4 Quoting, running, and reading the answer back

`kind: "evaluate"`, not the default `"simulate"`. Neither document has a `D(·)`
anywhere: `system_kind` is `nonlinear`, the analyses' time spans are `0 -> 0`,
and the whole answer is the observed graph. Dispatched as a simulation the engine
does not return something meaningless — it refuses the document outright with
`Invalid parameter 'src_E'`, which names the wrong thing entirely.

`POST /quote` needs no auth and no database; it is the pre-login preview.

The progress fraction is **phase**-weighted across eight `PreparePhase`s that
differ by four orders of magnitude in cost — `Rewrite` is milliseconds and
`GatedFetch` is tens of gigabytes off S3, so 77% is the fetch 16% in. A long
crawl near the end is the I/O, not a hang.

In [ ]:
TERMINAL = {"succeeded", "failed", "cancelled", "capped"}


def money(dollars):
    return "—" if dollars is None else (
        f"${dollars:.4f}" if 0 < abs(dollars) < 0.01 else f"${dollars:.2f}")


def clock(seconds):
    seconds = int(seconds)
    if seconds >= 3600:
        return f"{seconds // 3600}h{seconds % 3600 // 60:02d}m"
    return f"{seconds // 60}m{seconds % 60:02d}s" if seconds >= 60 else f"{seconds}s"


def quote(doc, observeds):
    """Price the run. No auth — `POST /quote` has no database."""
    routing = http_json("POST", f"{API}/quote", timeout=300.0,
                        json_body={"esm": doc, "kind": "evaluate", "observeds": observeds})
    option = routing.get("dispatchable")
    if not option:
        raise RuntimeError(f"no dispatchable backend: {routing.get('reason')}")
    e, sizing = option["estimate"], routing.get("sizing") or {}
    print(f"  backend    {option['backend']} (tier {routing['tier']})")
    print(f"  machine    {sizing.get('vcpus')} vCPU / {sizing.get('memory_mb')} MB")
    print(f"  predicted  {clock(e['resource_seconds'])}   cap {clock(e['max_resource_seconds'])}")
    print(f"  price      {money(e['price'])}")
    print(f"  why        {routing.get('reason', '')}")
    return e["price"]


def watch(session, run_id):
    """Follow a run to a terminal event, surviving a dropped connection.

    The stream replays everything already recorded before it streams, so a
    reconnect sees what it missed — including a terminal event that landed while
    we were disconnected. That is what makes reconnecting sufficient.
    """
    started = time.time()
    while True:
        try:
            with session.stream(f"/runs/{run_id}/events", timeout=240.0) as resp:
                for line in resp:
                    line = line.decode("utf-8", "replace").strip()
                    if not line.startswith("data:"):
                        continue
                    event = json.loads(line[5:]).get("kind") or {}
                    kind = event.get("type")
                    if kind == "progress":
                        f = event.get("fraction", 0.0)
                        print(f"\r  [{'#' * int(f * 40):<40}] {f * 100:5.1f}%  "
                              f"elapsed {clock(time.time() - started)}", end="", flush=True)
                    elif kind in ("queued", "started"):
                        print(f"  {kind}", flush=True)
                    elif kind in TERMINAL:
                        print()
                        return event
        except (HttpError, OAuthError, urllib.error.URLError, OSError, ValueError) as e:
            print(f"\n  (stream dropped: {e}; the run is server-side and unaffected)")
        run = session.get(f"/runs/{run_id}")
        if run["status"] in TERMINAL:
            return {"type": run["status"]}
        time.sleep(5)


def read_fields(session, dataset_id, names):
    """One 1-D array per name, out of the run's own Zarr store.

    An evaluate run writes `[eval(1), rcv_cells(52411)]`, so pinning every
    length-1 axis leaves exactly the receptor axis free — two free axes is a
    field, one is a line, and a line is what a map's value column is.
    """
    dataset = session.get(f"/datasets/{dataset_id}")
    pins = ",".join(f"{d['name']}:0" for d in dataset.get("dims", []) if d["size"] == 1)
    series = {}
    for name in names:
        query = {"var": name, "max_values": MAX_VALUES}
        if pins:
            query["at"] = pins
        field = session.get(f"/datasets/{dataset_id}/field?{urllib.parse.urlencode(query)}")
        axis = field["axes"][0]
        if len(field["axes"]) != 1:
            raise RuntimeError(f"{name}: {len(field['axes'])} free axes, expected 1")
        if axis["stride"] != 1:
            print(f"  ! {name} DECIMATED to every {axis['stride']}th of {axis['stored_size']}")
        series[name] = field["values"]
    return series


def run_and_read(session, doc, observeds, max_price):
    """Dispatch, watch, and bring the fields home."""
    run = session.post("/runs", {"esm": doc, "kind": "evaluate",
                                 "observeds": observeds, "max_price": max_price})
    print(f"  run {run['id']} — {run['status']} on {run['backend']}, {money(run['price'])}")
    outcome = watch(session, run["id"])
    if outcome["type"] != "succeeded":
        raise RuntimeError(f"run {run['id']} {outcome['type']}: {outcome.get('message', '')}")
    print(f"  succeeded in {clock(outcome.get('resource_seconds', 0))} of resource time")
    return run["id"], read_fields(session, outcome["dataset_id"], observeds)


def report_totals(doc, series):
    for key, name in totals_for(doc).items():
        print(f"  sum({name})".ljust(22) + repr(sum(series[name])))

## 1.5 The map

One rectangle per receptor cell, from the `rcv_W/S/E/N` the documents compute,
filled by concentration. The ISRM grid is variable-resolution — coarse over
rural country, fine over cities — so the mesh itself carries information that a
resampled raster would throw away, and drawing the real rectangles shows it.

Three choices worth stating, because each one could be made badly:

**One hue, light to dark.** Concentration is a *magnitude*, and magnitude gets a
sequential ramp. Not a rainbow: a rainbow implies category boundaries the data
does not have, and it is unreadable to a colorblind viewer.

**The colour scale is clipped at a percentile, and says so.** PM2.5 over this
grid is extremely skewed — a handful of cells beside large sources sit orders of
magnitude above the median — so a scale stretched to the true maximum renders
essentially the whole country as the lightest step and shows nothing. `clip_pct`
caps the ramp and the subtitle prints the real maximum, so the cells above the
cap are visibly the darkest without the number being hidden.

**Equal aspect.** These are Lambert-conformal metres in both directions; letting
the axes scale independently would be a distorted map.

In [ ]:
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.collections import PolyCollection
from matplotlib.colors import LinearSegmentedColormap

# Chart chrome, and one sequential blue ramp light -> dark.
SURFACE, INK, INK_2, MUTED, GRID = "#fcfcfb", "#0b0b0b", "#52514e", "#898781", "#e1e0d9"
SEQUENTIAL = LinearSegmentedColormap.from_list("esl_blue", [
    "#cde2fb", "#b7d3f6", "#9ec5f4", "#86b6ef", "#6da7ec", "#5598e7", "#3987e5",
    "#2a78d6", "#256abf", "#1c5cab", "#184f95", "#104281", "#0d366b",
])


def percentile(values, pct):
    """The `pct`th percentile, without requiring numpy."""
    ordered = sorted(values)
    if not ordered:
        return 0.0
    k = (len(ordered) - 1) * pct / 100.0
    lo, hi = int(k), min(int(k) + 1, len(ordered) - 1)
    return ordered[lo] + (ordered[hi] - ordered[lo]) * (k - lo)


def focus_bounds(series, value, share=0.90, pad=0.06):
    """A view box around the cells that carry `share` of the total.

    An emission layer covering one state still produces a field over the whole
    national grid, and cropping keeps the map about the data rather than about
    the grid's extent. Cells are taken in descending order until they account for
    `share` of the total, rather than thresholded at a fraction of the maximum,
    because a source–receptor field's tail is thin but very wide: almost every
    cell in the country sits above any small fraction of the peak, so a threshold
    rule crops nothing exactly when cropping is wanted most.

    **0.90 rather than 0.99 for the same reason.** The last tenth of the mass is
    spread over most of the continent — on a single-source test field, going from
    0.90 to 0.99 takes the box from roughly 1,700 km across to the full grid — so
    a high share quietly undoes the crop. Raise it to see more of the plume.
    """
    v = series[value]
    total = sum(v)
    if total <= 0:
        return None
    keep, running = [], 0.0
    for i in sorted(range(len(v)), key=lambda i: -v[i]):
        keep.append(i)
        running += v[i]
        if running >= total * share:
            break
    x0 = min(series["rcv_W"][i] for i in keep)
    x1 = max(series["rcv_E"][i] for i in keep)
    y0 = min(series["rcv_S"][i] for i in keep)
    y1 = max(series["rcv_N"][i] for i in keep)
    mx, my = (x1 - x0) * pad, (y1 - y0) * pad
    return x0 - mx, x1 + mx, y0 - my, y1 + my


def draw_map(series, value="TotalPM25", title="", subtitle="", units="µg/m³",
             clip_pct=99.0, bounds=None, figsize=None, height=6.4):
    """The receptor grid as a choropleth: one rectangle per cell."""
    W, S, E, N = (series[f"rcv_{k}"] for k in ("W", "S", "E", "N"))
    v = series[value]
    rects = [((W[i], S[i]), (E[i], S[i]), (E[i], N[i]), (W[i], N[i])) for i in range(len(v))]

    vmax = percentile(v, clip_pct) or max(v) or 1.0
    x0, x1, y0, y1 = bounds or (min(W), max(E), min(S), max(N))

    # The figure follows the MAP's shape. These are metres in both directions, so
    # the axes are equal-aspect and their width is decided by the extent, not by
    # us — a fixed landscape figure around a square crop is mostly margin.
    if figsize is None:
        span = height * (x1 - x0) / (y1 - y0) if y1 > y0 else height
        figsize = (min(13.0, max(4.5, span)) + 1.8, height)

    fig, ax = plt.subplots(figsize=figsize, dpi=150)
    fig.patch.set_facecolor(SURFACE)
    ax.set_facecolor(SURFACE)

    # No edge colour: at 52,411 cells a stroke per rectangle is most of the ink
    # on the page, and the boundaries it draws are the grid's, not the data's.
    mesh = PolyCollection(rects, array=v, cmap=SEQUENTIAL, linewidths=0.0,
                          edgecolors="none", rasterized=True)
    mesh.set_clim(0.0, vmax)
    ax.add_collection(mesh)

    ax.set_xlim(x0, x1)
    ax.set_ylim(y0, y1)
    ax.set_aspect("equal")

    bar = fig.colorbar(mesh, ax=ax, fraction=0.03, pad=0.02,
                       extend="max" if max(v) > vmax else "neither")
    bar.set_label(f"{value} ({units})", color=INK_2, fontsize=9)
    bar.ax.tick_params(colors=MUTED, labelsize=8, length=3)
    bar.outline.set_visible(False)

    # Two lines, and the title's padding clears both: provenance is the caller's
    # sentence, and how the colour scale was cut is ours. One line of the two
    # combined overflows the figure at any ordinary width.
    ax.set_title(title or value, color=INK, fontsize=13, pad=46, loc="left")
    caption = (f"{subtitle or f'{len(v):,} receptor cells'}\n"
               f"{len(v):,} cells · scale clipped at the {clip_pct:g}th percentile "
               f"({vmax:.3g} {units}); true maximum {max(v):.3g} {units}")
    ax.text(0.0, 1.0, caption, transform=mpl.transforms.offset_copy(
        ax.transAxes, fig=fig, x=0, y=10, units="points"),
        color=MUTED, fontsize=8, va="bottom")

    ax.set_xlabel("easting (m, Lambert conformal)", color=INK_2, fontsize=9)
    ax.set_ylabel("northing (m, Lambert conformal)", color=INK_2, fontsize=9)
    for side in ("top", "right"):
        ax.spines[side].set_visible(False)
    for side in ("left", "bottom"):
        ax.spines[side].set_color(GRID)
    ax.tick_params(colors=MUTED, labelsize=8, length=3)
    fig.tight_layout()
    return fig, ax

## 1.6 Preflight: can the runner actually read this?

**The runner reads `s3://` anonymously.** EarthSciIO's `s3` transport is a
bucket-to-regional-HTTPS rewriter — its own module note says "a public bucket
needs no AWS SDK, no SigV4, no credentials", and SigV4 is listed as a *future*
resolver. `store_access` / `store_options`, which are where a loader could ask
for a signed read, are **ignored by whole-file readers**: they exist for
store-backed (Zarr) reads only. So a `.zip` behind a private bucket is
unreadable by a dispatched run no matter what IAM says.

That failure costs a run to discover — it surfaces as `HTTP 403` during extent
discovery, minutes in. This does the same fetch the runner will do, from here,
for nothing. It is not a guess about the allowlist; it is the actual GET.

In [ ]:
def anonymous_url(url, region="us-east-2"):
    """The URL EarthSciIO's `s3` transport will really request.

    `s3://<bucket>/<key>` becomes virtual-hosted regional HTTPS. The region
    follows the runner's own chain — `EARTHSCI_S3_REGION`, then `AWS_REGION`,
    then a compiled `us-east-2` — and the job definitions pin it to us-east-2.
    """
    if not url.startswith("s3://"):
        return url
    bucket, _, key = url[len("s3://"):].partition("/")
    return f"https://{bucket}.s3.{region}.amazonaws.com/{key}"


def preflight_readable(doc, signed_buckets=(), raise_on_fail=True):
    """Fetch each loader URL the way the runner will, before paying for a run.

    Only loaders the engine will actually build are checked — the same two
    conditions the server's own sweep uses, an `esio_format` and a
    `url_template` — and a dated template (one carrying `[year]`-style
    placeholders) is skipped rather than guessed at.

    `signed_buckets` are the ones the DEPLOYMENT reads with credentials
    (`EARTHSCI_S3_SIGNED_BUCKETS` on the runner). We cannot probe those from
    here — an anonymous GET of a private bucket is 403 whether or not the runner
    can read it — so they are reported as unprobed rather than failed. Probing
    them anonymously and refusing would block exactly the path that works.
    """
    signed = {b.lower() for b in signed_buckets}
    problems = []
    for name, source in (doc.get("data_sources") or {}).items():
        url = (source.get("source") or {}).get("url_template", "")
        if not url or not (source.get("metadata") or {}).get("esio_format"):
            continue
        if "[" in url:
            print(f"  {name:14} skipped (dated template)")
            continue
        if url.startswith("s3://") and url[5:].split("/")[0].lower() in signed:
            print(f"  {name:14} —   the runner reads this bucket signed (not probed)")
            continue

        # A store-backed loader's URL is a PREFIX, not an object — GET it and S3
        # says 404 whether or not the store is there. So probe the store's own
        # metadata key instead, accepting any of the three spellings (Zarr v3
        # writes `zarr.json`; a zarr-python v2 store has `.zmetadata` and always
        # `.zgroup`). Whole-file formats are the easy case: the URL is the object.
        fmt = (source.get("metadata") or {}).get("esio_format")
        base = anonymous_url(url).rstrip("/")
        probes = ([f"{base}/{k}" for k in ("zarr.json", ".zmetadata", ".zgroup")]
                  if fmt == "zarr" else [anonymous_url(url)])

        status = None
        for probe in probes:
            try:
                req = urllib.request.Request(probe, headers={"Range": "bytes=0-99"})
                with urllib.request.urlopen(req, timeout=30) as resp:
                    status = resp.status
                    break
            except urllib.error.HTTPError as e:
                status = e.code
                if e.code in (405, 416):        # dislikes the Range, not the object
                    break
            except Exception as e:              # DNS, TLS, timeout
                print(f"  {name:14} {type(e).__name__}: {e}  (transport, may be flaky)")
                status = None
                break

        if status is None:
            continue
        ok = status < 400 or status in (405, 416)
        print(f"  {name:14} {status} " + ("readable" if ok else "UNREADABLE"))
        if not ok:
            problems.append((name, probes[-1], f"HTTP {status}"))

    if problems and raise_on_fail:
        detail = "\n".join(f"  {n}: {why} on {u}" for n, u, why in problems)
        raise RuntimeError(
            "a loader the run will build is not anonymously readable, and the runner has "
            "no other way to read it:\n" + detail +
            "\n\nA private bucket cannot be fixed with IAM here — the s3 transport does not "
            "sign. Put the bytes somewhere public that the deployment allowlists."
        )
    return not problems

---
# 2. `isrm_point.esm` — the EGU point inventory

The EPA 2016fd alpha point-source FF10 inventory through the ISRM, with ASME
plume rise splitting every record's mass across the model's three emission
layers. Its analysis is `isrm_demo`.

### Choose the scale before you run it

`RECORDS` is the one knob. It is set low here on purpose — a notebook cell that
takes fifty minutes by default is not a good notebook cell — but the full
inventory is the run that reproduces the published totals.

| `RECORDS` | records | roughly | `sum(deathsK)` should be |
|---|---|---|---|
| `200` | 200 | a few minutes | `49.11639491165982` |
| `2000` | 2,000 | ~10 minutes | `363.47671096747285` |
| `None` | 43,650 (all) | **~50 minutes** | `7022.724781368745` |

Those reference values are this repo's own `run-*/results*.json`, produced by the
Rust shim driving the same engine in-process. Expect agreement to about `1e-15`
rather than bit-identity: the shims sum with Kahan compensation and this folds
naively, a difference the repo README records at ~`2.9e-13` on *bit-identical*
fields. A disagreement bigger than that is a real one.

Either way the price is the same — **$0.04**. The estimator floors a static
evaluation at a fixed duration and cannot yet tell the two apart, so what
`RECORDS` buys you is wall-clock, not money.

In [ ]:
RECORDS = 200          # <- None for the full 43,650-record inventory (~50 min)

point_doc, point_analysis, point_observeds = prepare(
    REPO / "isrm_point.esm", records=RECORDS, analysis_id="isrm_demo")

In [ ]:
preflight_readable(point_doc)
point_price = quote(point_doc, point_observeds)

In [ ]:
point_run_id, point = run_and_read(session, point_doc, point_observeds, point_price)
report_totals(point_doc, point)

Point sources are spread across the whole country, so this one is drawn national.

In [ ]:
fig, ax = draw_map(
    point,
    title="PM2.5 from EGU point sources, through the InMAP ISRM",
    subtitle=(f"isrm_point.esm · {'all 43,650' if RECORDS is None else f'first {RECORDS:,}'} "
              f"emission records · EarthSciLab run {point_run_id[:8]}"),
)
plt.show()

## 3.1 A local file cannot be dispatched, and the fix that makes the upload work

```json
"Polygon_Emis": { "source": { "url_template": "file://data/polygon_emissions_17.zip" } }
```

That is a local path, and it fails on the server twice over: `file://` is
deliberately not on the run allowlist (a dispatched run binds the *document's
own* loaders, which makes a document a fetch instruction, so a loader pointing at
`file:///proc/self/environ` would be a worker reading its own secrets into a
dataset the caller can download), and the bytes are on this machine anyway.

So we upload the layer as an EarthSciLab **dataset** and re-point the loader.

### Why that used to fail, and what changed

The upload always worked. The *read back* did not, and it took a run to find out:

```
extent discovery for 'ISRM.poly_emis': all sources failed for
s3://earthscilab-run-outputs-…/datasets/…/polygon_emissions_17.zip: HTTP 403
```

EarthSciIO's `s3://` transport is an anonymous `s3://` → regional-HTTPS
rewriter — "a public bucket needs no AWS SDK, no SigV4, no credentials". That is
the right default: inferring "we can authenticate" from ambient credentials signs
a read of a *public* bucket and turns it into a 403, which is a bug this codebase
has already fixed once. But it left private buckets with no path at all through
the cache, and **every whole-file format reads through that path** — `shapefile`,
`ff10`, `netcdf`, `geotiff`. `store_access` / `store_options`, where a loader
would ask for a signed read, are store-backed (Zarr) only. So the job role's
`s3:GetObject` on `datasets/*` was necessary and not sufficient: the transport
never signed, whatever IAM said.

**The fix is upstream, in EarthSciIO:** a bucket named in
`EARTHSCI_S3_SIGNED_BUCKETS` is fetched signed, through `object_store`'s AWS
client — the same SigV4 and credential chain the direct Zarr path already uses.
Everything else stays anonymous byte for byte, and there is deliberately no
wildcard, so naming one bucket can never make a public read start signing.
EarthSciLab names exactly one: its own dataset store, derived from the run's own
output target on Fly and from `aws_s3_bucket.outputs.id` on Batch.

Nothing in this document changes, and no credential goes anywhere near it.

In [ ]:
def upload_file_dataset(session, path, esio_format):
    """Put a local file in the dataset store and return its committed record.

    Three calls, and the middle one is the bytes. `POST /datasets` mints the row
    and says where to write; the server derives everything it can rather than
    trusting the client — `format` and `origin` are the two exceptions, because
    deriving them would mean opening bytes this API has decided not to open.
    """
    path = Path(path)
    payload = path.read_bytes()

    created = session.post("/datasets", {"format": esio_format, "origin": "upload"})
    if created["upload"]["mode"] != "proxy":
        raise RuntimeError(
            f"this deployment wants a {created['upload']['mode']!r} upload, not a proxied one; "
            "see the UploadTarget union in the API docs")
    if len(payload) > created["max_object_bytes"]:
        raise RuntimeError(f"{path.name} is {len(payload):,} B, over the "
                           f"{created['max_object_bytes']:,} B per-object ceiling")

    target = f"{created['upload']['url']}?key={urllib.parse.quote(path.name)}"
    session.put_bytes(target, payload)
    dataset = session.post(f"/datasets/{created['id']}/commit")

    print(f"  uploaded {path.name} ({len(payload):,} B) as dataset {dataset['id']}")
    if dataset.get("expires_at"):
        print(f"  unsaved — kept until {dataset['expires_at']} "
              f"(POST /datasets/{dataset['id']}/save to keep it)")
    return dataset


def loader_url_for(dataset):
    """Where a document's `url_template` should point at this dataset.

    Only Zarr is store-backed — many objects under one prefix — so only Zarr
    addresses the directory. Every other format is one blob, named by its key.
    """
    base = dataset["store_url"].rstrip("/")
    return base if dataset.get("format") == "zarr" else f"{base}/{dataset['object_key']}"

## 3.2 Build the example layer

`data/make_polygon_layer.py` is the step a person does in Python before pointing
a document at the result, and it is deliberately ordinary: fetch a public
boundary file, keep the polygons you care about, attach an emission column, write
a shapefile. The default is Illinois' 102 counties at a uniform 1 short
ton/yr/km² — an **example** quantity, and the document says so. The point of the
polygon path is the geometry, not the inventory.

It needs `pyshp`, and nothing else.

In [ ]:
POLYGON_ZIP = REPO / "data" / "polygon_emissions_17.zip"

if not POLYGON_ZIP.is_file():
    subprocess.run([sys.executable, str(REPO / "data" / "make_polygon_layer.py")], check=True)
print(f"{POLYGON_ZIP.name}: {POLYGON_ZIP.stat().st_size:,} bytes")

In [ ]:
polygon_dataset = upload_file_dataset(session, POLYGON_ZIP, "shapefile")
polygon_url = loader_url_for(polygon_dataset)
print("  loader url:", polygon_url)

## 3.3 Run it

One line of the document changes — the loader's URL. Everything else, including
the `select` range if you set `RECORDS`, works exactly as it did for the point
document.

All 102 polygons is the whole layer and it is small, so there is no reason to
truncate; the cost here is the gated SR fetch for the source cells Illinois
covers, not the record count.

In [ ]:
POLY_RECORDS = None    # all 102 counties; set an int to truncate

# Where the runner reads the layer from. `None` uploads it as an EarthSciLab
# dataset, which is the intended path: the runner reads its own dataset store
# signed (see 3.1). Set a public, allowlisted URL instead to bypass the upload.
POLYGON_URL = None

polygon_doc, polygon_analysis, polygon_observeds = prepare(
    REPO / "isrm_polygon.esm", records=POLY_RECORDS, analysis_id="isrm_polygon_demo")

signed = []
if POLYGON_URL:
    url = POLYGON_URL
else:
    dataset = upload_file_dataset(session, POLYGON_ZIP, "shapefile")
    url = loader_url_for(dataset)
    # The store the runner signs for. Named here so the preflight does not probe
    # it anonymously and refuse the one path that works.
    signed = [dataset["store_url"].removeprefix("s3://").split("/")[0]]

polygon_doc["data_sources"]["Polygon_Emis"]["source"]["url_template"] = url
print("  Polygon_Emis ->", url)

# Costs nothing, and is the difference between finding out now and finding out
# several minutes into a run you paid for.
preflight_readable(polygon_doc, signed_buckets=signed)

In [ ]:
polygon_price = quote(polygon_doc, polygon_observeds)

In [ ]:
polygon_run_id, polygon = run_and_read(
    session, polygon_doc, polygon_observeds, polygon_price)
report_totals(polygon_doc, polygon)

Illinois-only emissions over a national grid: drawn at full extent this map would
be a small bright patch in an ocean of zero, so `focus_bounds` crops to the cells
that actually carry signal. Pass `bounds=None` to see the whole country and how
far downwind the plume reaches.

In [ ]:
fig, ax = draw_map(
    polygon,
    title="PM2.5 from an Illinois county area-source layer",
    subtitle=(f"isrm_polygon.esm · 102 polygons at 1 t/yr/km² (an example rate) · "
              f"EarthSciLab run {polygon_run_id[:8]}"),
    bounds=focus_bounds(polygon, "TotalPM25"),
)
plt.show()

## 3.4 Cleaning up

The uploaded layer is 37 KB and storage is metered from byte one, so this is
housekeeping rather than economy — but an unsaved dataset also carries an
`expires_at`, and a re-run of section 3 after that date needs a fresh upload
anyway.

In [ ]:
# session.post(f"/datasets/{polygon_dataset['id']}/save")     # keep it indefinitely
# http_json("DELETE", f"{API}/datasets/{polygon_dataset['id']}", headers=session.headers())
[d["id"] for d in session.get("/datasets")][:5]